In [9]:
# Chat model

import getpass
import os

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

# Embeddings model

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# Vector store

from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="neuro_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  
)


In [10]:
from langchain.prompts import PromptTemplate

SYSTEM_RULES = {
    "rules": [
        "Neurotechnology implants have become popular by 2075.",
        "Memory manipulation is technically possible but ethically controversial.",
        "Each year, one major neurotech-related event happens.",
        "Players always receive exactly 3 choices."
    ],
    "constraints": [
        "Scenarios must stay realistic for neurotechnology research.",
    ]
}


BASE_PROMPT = """
Follow these fixed system rules (do not change them):
{system_rules}

Context:
{context}
"""

scenario_writer_prompt = PromptTemplate.from_template(
    BASE_PROMPT + """
    You are the Scenario Writer.
    Write a ~80 word scenario describing a neurotechnology-related event in the year {year}.
    Use ONLY the provided Context above. Do NOT invent sources, names, or events not present in Context.
    The "citations" field is pre-filled with the actual filenames from the retrieved context; do not change it.

    Output JSON:
    {{
        "year": {year},
        "scenario_text": "...",
        "citations": [{citations}]
    }}
    """
)

choice_maker_prompt = PromptTemplate.from_template(
    BASE_PROMPT + """
    You are the Choice Maker.
    Base your decisions ONLY on the SCENARIO below and the Context.
    
    Scenario:
    {scenario}

    Output JSON:
    {{
        "choices": [
            {{"id": 1, "text": "..."}},
            {{"id": 2, "text": "..."}},
            {{"id": 3, "text": "..."}}
        ]
    }}
    """
)

outcome_updater_prompt = PromptTemplate.from_template(
    BASE_PROMPT + """
    You are the Outcome Generator.
    Given the scenario, the available choices, and the player's choice with id {choice_id}, describe the outcome.

    Scenario:
    {scenario}

    Choices:
    {choices}

    output JSON:
    {{
        "choice_id": {choice_id},
        "outcome_text": "..."
    }}
    """
)


In [11]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, START
from typing_extensions import List, TypedDict
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader

DATA_DIR = Path("./neurotech")

docs = []
for path in DATA_DIR.rglob("*"):
    suf = path.suffix.lower()
    if suf == ".pdf":
        loaded = PyPDFLoader(str(path)).load()
    elif suf == ".docx":
        loaded = Docx2txtLoader(str(path)).load()
    else:
        continue
    for d in loaded:
        d.metadata = d.metadata or {}
        d.metadata["source"] = path.name
    docs.extend(loaded)


text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

all_splits = [c for c in all_splits if c.page_content and c.page_content.strip()]

_ = vector_store.add_documents(all_splits)


class State(TypedDict):
    question: str
    context: List[Document]
    answer: str
    role: str
    scenario: str
    choices: List[dict]
    choice_id: int
    year: int


def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    role = state["role"]
    year = state.get("year", 2075)

    if role == "scenario":
        sources = []
        for doc in state["context"]:
            src = (doc.metadata or {}).get("source")
            if src and src not in sources:
                sources.append(src)
        if not sources:
            sources = ["(no_source_found)"]
            
        citations_literal = ", ".join([f'"{s}"' for s in sources])
        
        prompt = scenario_writer_prompt
        messages = prompt.format(
            context=docs_content,
            year=year,
            system_rules=SYSTEM_RULES,
            citations=citations_literal
        )
        response = llm.invoke(messages)
        return {"answer": response.content, "scenario": response.content}

    elif role == "choices":
        prompt = choice_maker_prompt
        messages = prompt.format(context=docs_content, year=year, system_rules=SYSTEM_RULES, scenario=state.get("scenario", ""))
        response = llm.invoke(messages)
        return {"answer": response.content, "choices": response.content}

    elif role == "outcome":
        prompt = outcome_updater_prompt
        messages = prompt.format(
            context=docs_content,
            system_rules=SYSTEM_RULES,
            scenario=state.get("scenario", ""),
            choices=state.get("choices", ""),
            choice_id=state.get("choice_id", None)
        )
        response = llm.invoke(messages)
        return {"answer": response.content}

    else:
        raise ValueError(f"Unknown role: {role}")


graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

In [12]:
scenario_response = graph.invoke({"question": "Make a scenario", "role": "scenario", "year": 2076})
print("Scenario:", scenario_response["answer"])
scenario = scenario_response["scenario"]

choices_response = graph.invoke({"question": "What are the possible choices?", "role": "choices", "scenario": scenario})
print("Choices:", choices_response["answer"])
choices = choices_response["choices"]

response = graph.invoke({"question": "What happens next?", "role": "outcome", "scenario": scenario, "choices": choices,"choice_id": 2})
print("Outcome:", response["answer"])



Scenario: ```json
{
    "year": 2076,
    "scenario_text": "In 2076, the ethical controversy surrounding memory manipulation intensified following a high-profile public case. Claude, whose abrupt decision to close his decades-old organization and move countries led to significant personal and interpersonal repercussions, became the center of a legal debate. Colleagues perceived his actions as selfish, and his wife struggled to comprehend the change, highlighting the harm to relationships. While some argued his memory alteration was an autonomously constructed narrative (Wiley et al., 1998), others, citing Tan and Lim (2020), warned against forceful alterations of one’s self-narrative, stressing the potential negative consequences.",
    "citations": ["fpsyg-14-1282634.pdf", "Neuromodulation_and_memory_exploring_ethical_ramif.pdf"]
}
```
Choices: ```json
{
    "choices": [
        {
            "id": 1,
            "text": "Push for a global moratorium on all non-therapeutic memory mani